# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## 1. Ranked actions + reason codes

My baseline prioritizes content that is both stale and visible.

A page is considered stale when `days_since_last_update >= 180` and visible when `impressions_90d >= 500`.

The priority score is the observed `impressions_90d` value when both conditions are satisfied. This places higher-exposure stale pages earlier in the review queue.

### Reason code

`stale_visible` — the page has not been updated for at least 180 days and has at least 500 observed impressions in the 90-day window.

### Action

`refresh_first` — send the page to an SEO specialist or content editor for human review and possible refresh.

Pages that do not meet both conditions receive the `monitor` action.

### Archetype-to-action mapping

| Archetype | Condition | Recommended action |
|---|---|---|
| Stale + visible | stale and impressions >= 500 | Refresh first |
| Recent + visible | not stale and impressions >= 500 | Monitor |
| Stale + low visibility | stale and impressions < 500 | Monitor / investigate |
| Recent + low visibility | not stale and impressions < 500 | Monitor |

The ranking is a decision-support tool and does not automatically determine what change should be made to a page.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Define baseline signals
df["stale"] = df["days_since_last_update"] >= 180
df["visible"] = df["impressions_90d"] >= 500

# Priority score
df["baseline_score"] = (
    df["stale"].astype(int)
    * df["visible"].astype(int)
    * df["impressions_90d"]
)

# Reason code
df["reason_code"] = np.where(
    df["stale"] & df["visible"],
    "stale_visible",
    "not_priority"
)

# Action
df["action"] = np.where(
    df["stale"] & df["visible"],
    "refresh_first",
    "monitor"
)

# Ranked queue
queue = df.sort_values(
    "baseline_score",
    ascending=False
).copy()

print("Priority candidates:", (queue["baseline_score"] > 0).sum())

display(
    queue[
        [
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 4.68 MiB/s, done.
Resolving deltas: 100% (153/153), done.
/content/flyrank-ml-internship-starter
Priority candidates: 17


,content_id,days_since_last_update,impressions_90d,avg_position,ctr,baseline_score,reason_code,action
16751,content_cf56e2e2e282,194,61678,19.7,0.15,61678,stale_visible,refresh_first
16514,content_7368877ea310,194,59472,24.8,0.13,59472,stale_visible,refresh_first
7021,content_1bfaa38ff26c,194,25715,22.2,0.23,25715,stale_visible,refresh_first
21268,content_0a91db491d14,193,13299,10.5,0.49,13299,stale_visible,refresh_first
11489,content_5feee3994adb,194,7812,39.0,0.01,7812,stale_visible,refresh_first
12045,content_c2d929d83eaa,193,7558,17.9,0.20,7558,stale_visible,refresh_first
698,content_b16bd7307b39,194,4590,31.0,0.00,4590,stale_visible,refresh_first
5327,content_fe16a55cd13d,194,4556,16.4,0.33,4556,stale_visible,refresh_first
26810,content_ecb6215e79fd,194,4429,25.3,0.38,4429,stale_visible,refresh_first
20837,content_928af3e22c80,193,1697,15.8,0.12,1697,stale_visible,refresh_first


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

### Intended use

The playbook is intended for SEO specialists and content editors who need to decide which content pages should receive attention first when editorial resources are limited.

The ranked queue uses observed historical signals to organize human review.

### Decay / refresh insight

The baseline uses content staleness as a prioritization signal because older content that has not been updated may deserve review when it still has meaningful search visibility.

This relationship is observational. It does not prove that refreshing a page will cause better rankings, traffic, clicks, or engagement.

### Limits

The playbook cannot:

- predict Google's ranking algorithm;
- guarantee search or traffic improvements;
- prove that refreshing a page causes better performance;
- determine the correct editorial change automatically;
- replace SEO or editorial judgment.

The Week-5 Random Forest did not outperform the simpler baseline on the evaluated Precision@K metrics. Therefore, the simpler rule remains the practical baseline for this playbook.

The output should be treated as directional decision-support rather than a guaranteed prediction.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review + the no-go list

### Human review rules

Every `refresh_first` recommendation must be reviewed by an SEO specialist or content editor before any change is made.

The reviewer should check:

1. Whether the page is still relevant to its intended search intent.
2. Whether the existing information is accurate and useful.
3. Whether the page has business or editorial context that is not represented in the dataset.
4. Whether its observed search visibility is meaningful.
5. Whether refreshing is preferable to monitoring, rewriting, merging, or leaving the page unchanged.

### Cost / value thinking

The main cost of a false positive is editorial time spent reviewing or refreshing a page that does not have meaningful improvement potential.

The cost of a false negative is that a page that may deserve attention remains lower in the queue.

Because editorial time is limited, the ranking can help reviewers spend their effort on higher-exposure candidates first.

The score does not estimate financial return. It is only a prioritization measure based on observed search exposure.

### No-go list

The system should NOT automatically:

- publish content changes;
- delete or prune pages;
- rewrite content;
- change titles or metadata;
- change search-intent classifications;
- claim that a refresh will improve rankings;
- treat the score as proof of causal impact;
- make a final editorial decision without human review.

The playbook is a prioritization and decision-support tool, not an autonomous content-management system.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring / retrain triggers

### Monitoring

The playbook should be periodically reviewed for changes in:

- the number of `stale_visible` pages;
- the distribution of impressions among recommended pages;
- the distribution of days since last update;
- the characteristics of pages entering the priority queue;
- Precision@K when a later observed outcome window becomes available.

### Review triggers

The baseline should be reconsidered if:

- measured Precision@K declines consistently on newly observed data;
- the number or characteristics of recommended pages changes substantially;
- the distributions of staleness or impressions shift substantially;
- reviewers repeatedly identify the same type of weak recommendation.

### Retrain / model reconsideration trigger

A more complex ML model should only be reconsidered if new evidence shows that it consistently improves on the simple baseline under the same honest validation design.

Any future model should use a client-grouped or time-aware validation strategy and repeat the leakage audit.

Model complexity should not be increased simply because a more complex model is available.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## 5. Exports for the paper

The notebook exports the ranked action queue that will support the recommendations section of the research paper.

The queue is regenerated by the notebook rather than committed as a raw data file, keeping the workflow reproducible and following the project's data-handling rules.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("work/outputs", exist_ok=True)

paper_queue = queue[
    [
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

output_path = "work/outputs/baseline_action_score.csv"

paper_queue.to_csv(
    output_path,
    index=False
)

print("Exported:", output_path)
print("Rows:", len(paper_queue))

display(paper_queue.head(20))

Exported: work/outputs/baseline_action_score.csv
Rows: 30000


,content_id,days_since_last_update,impressions_90d,avg_position,ctr,baseline_score,reason_code,action
16751,content_cf56e2e2e282,194,61678,19.7,0.15,61678,stale_visible,refresh_first
16514,content_7368877ea310,194,59472,24.8,0.13,59472,stale_visible,refresh_first
7021,content_1bfaa38ff26c,194,25715,22.2,0.23,25715,stale_visible,refresh_first
21268,content_0a91db491d14,193,13299,10.5,0.49,13299,stale_visible,refresh_first
11489,content_5feee3994adb,194,7812,39.0,0.01,7812,stale_visible,refresh_first
12045,content_c2d929d83eaa,193,7558,17.9,0.20,7558,stale_visible,refresh_first
698,content_b16bd7307b39,194,4590,31.0,0.00,4590,stale_visible,refresh_first
5327,content_fe16a55cd13d,194,4556,16.4,0.33,4556,stale_visible,refresh_first
26810,content_ecb6215e79fd,194,4429,25.3,0.38,4429,stale_visible,refresh_first
20837,content_928af3e22c80,193,1697,15.8,0.12,1697,stale_visible,refresh_first


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.